## Required Work

The cleaning in this notebook has two parts:

- Implement the cleaning recommendations you decided to adopt, including any overrides you need (for example, overriding a type assignment from the tool).
- Implement business rules to identify valid data that can move forward to featurization.

Use one method per step and apply them in sequence, as shown in the example. The business rules are shown below


- Identify loans as distressed or not distressed based on loan status and the interpretation of loan status. A non-distressed loan is one that is active and has not missed a payment.
- Identify invalid loans (loan status, bad dates, null first-disbursement dates).
- Drop attributes you do not need.
- Change types for fields that need type conversion, using `df_tags` as the authoritative source.
- Handle missing values for categorical attributes.
- Handle missing values for numeric attributes.

# Explanation of Loan Statuses Based on Gemini Research

This is the loan status description avaialble from Gemini
------------------------------
## 🟢 Active & Good Standing Statuses
These loans are operational and behaving exactly as intended.

* CURRENT ($54,435$ loans): The borrower is paying on time; the account is completely up to date.
* IN DEFERMENT ($100$ loans): The SBA or the lender has granted a temporary, authorized pause on monthly payments (often used during economic hardships or disasters). [1, 2] 

## 🔵 Successfully Closed Statuses
These loans have reached the end of their lifecycle with no losses incurred.

* PREPAID IN FULL ($34,428$ loans): The borrower paid off the entire loan balance early, ahead of the original maturity date.
* PAID IN FULL ($717$ loans): The borrower successfully completed their regular payment schedule and paid off the loan naturally. [1, 3] 

## 🟡 Delinquent Statuses (Past Due)
These loans are actively falling behind on their payments, broken down by exactly how many monthly cycles have been missed.

* 1 MONTH PAST DUE ($150$ loans) to 9 MONTHS PAST DUE ($7$ loans): The borrower has missed sequential monthly payments.
* >9 MONTHS PAST DUE ($12$ loans): Severe long-term delinquency where the borrower has missed 10 or more consecutive payments but the loan hasn't been officially written off yet. [1] 

## 🔴 Distress, Liquidation, & Default Statuses
These statuses indicate that a business has failed to pay, forcing the lender and the SBA to step in to recover funds.

* IN CATCH-UP ($1,061$ loans): The borrower fell behind but has entered into an approved agreement or payment plan to pay off the past-due balance. [1] 
* PURCH(NOT C/O) [Purchased, Not Charged Off] ($597$ loans): The borrower defaulted, so the SBA "purchased" the guaranteed portion of the loan from the commercial bank. However, it is not yet charged off because active collection efforts or asset liquidations are still ongoing. [4, 5] 
* PAID IN FULL (LIQ) [Liquidation] ($669$ loans): The loan was fully paid off, but only because the lender seized and sold off the business's collateral (real estate, equipment, inventory) to cover the debt. [4] 
* CHARGED-OFF ($910$ loans): The borrower defaulted, all collateral has been liquidated, and the SBA has officially written the remaining balance off as an uncollectible loss. [4] 
* ASSET SALE ($2$ loans): The non-performing loan note itself was sold at a discount to an outside debt buyer/investor to get it off the books. [5] 

## ⚪ Non-Origination Statuses
These applications never actually became live loans.

* CANCELED ($14,302$ loans): The loan approval was withdrawn or voided before closing docs were finalized (either by the borrower or the lender).
* NOT FUNDED ($8,261$ loans): The loan process was finalized and approved, but the capital was never successfully disbursed or accepted by the borrower. [3, 6] 


## The Intelligence Gathered

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from dd_cleaner.notebook_utils import (
    init_notebook_session, 
    get_raw_data, 
    get_cleaned_data, 
    get_tagged_entities,
    get_attributes_by_tag
)

# Initialize the session using the actual workspace root.
# When running from the notebooks/ directory, the project root sits one level up.
workspace_root = Path.cwd().resolve()
if not (workspace_root / "config.yaml").exists():
    workspace_root = workspace_root.parent

coord, artifacts_df = init_notebook_session(str(workspace_root))

# Display the artifacts table
artifacts_df

# 1. Initialize session and load the 'Clean Baseline'
# Point to the directory containing the 'data/' folder
coord, _ = init_notebook_session(str(workspace_root))
df = get_cleaned_data(coord)

print(f"Loaded Baseline: {df.shape[0]} rows, {df.shape[1]} columns")


✅ Notebook session initialized for workspace: /home/rajiv/programming/kmds_migration/sba_migration

Available Artifacts:

Artifact Name                                  File Name  \
0                         Raw Data                          sba_loans_raw.csv   
1                     Cleaned Data                    sba_loans_raw_clean.csv   
2                User Cleaned Data             sba_loans_raw_user_cleaned.csv   
3             Tagged Entities (DD)         sba_loans_raw_analysis_results.csv   
4  Cleaning Recommendations Report                cleaning_recommendations.md   
5                 Profiling Report          sba_loans_raw_profiling_report.md   
6                   Handshake File  sba_loans_raw_parser_cleaner_handshake.md   
7                  Quarantine File               sba_loans_raw_quarantine.csv   
8               Metadata Authority           sba_loans_raw_metadata_table.csv   

                                            Location  Exists  
0                             data/sba_loans_raw.csv    True  
1            data/dd_cleaner/sba_loans_raw_clean.csv    True  
2     data/dd_cleaner/sba_loans_raw_user_cleaned.csv    True  
3  documents/dd_analysis_results/sba_loans_raw_an...    True  
4   documents/dd_cleaner/cleaning_recommendations.md    True  
5  documents/dd_cleaner/sba_loans_raw_profiling_r...    True  
6  documents/dd_cleaner/sba_loans_raw_parser_clea...    True  
7       data/quarantine/sba_loans_raw_quarantine.csv   False  
8   data/dd_cleaner/sba_loans_raw_metadata_table.csv   False

✅ Notebook session initialized for workspace: /home/rajiv/programming/kmds_migration/sba_migration

Available Artifacts:

Artifact Name                                  File Name  \
0                         Raw Data                          sba_loans_raw.csv   
1                     Cleaned Data                    sba_loans_raw_clean.csv   
2                User Cleaned Data             sba_loans_raw_user_cleaned.csv   
3             Tagged Entities (DD)         sba_loans_raw_analysis_results.csv   
4  Cleaning Recommendations Report                cleaning_recommendations.md   
5                 Profiling Report          sba_loans_raw_profiling_report.md   
6                   Handshake File  sba_loans_raw_parser_cleaner_handshake.md   
7                  Quarantine File               sba_loans_raw_quarantine.csv   
8               Metadata Authority           sba_loans_raw_metadata_table.csv   

                                            Location  Exists  
0                             data/sba_loans_raw.csv    True  
1            data/dd_cleaner/sba_loans_raw_clean.csv    True  
2     data/dd_cleaner/sba_loans_raw_user_cleaned.csv    True  
3  documents/dd_analysis_results/sba_loans_raw_an...    True  
4   documents/dd_cleaner/cleaning_recommendations.md    True  
5  documents/dd_cleaner/sba_loans_raw_profiling_r...    True  
6  documents/dd_cleaner/sba_loans_raw_parser_clea...    True  
7       data/quarantine/sba_loans_raw_quarantine.csv   False  
8   data/dd_cleaner/sba_loans_raw_metadata_table.csv   False

Loaded Baseline: 388338 rows, 42 columns


/home/rajiv/programming/kmds_migration/sba_migration/.venv/lib/python3.13/site-packages/dd_cleaner/notebook_utils.py:124: DtypeWarning: Columns (0: ChargeOffDate) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path)


## The Tagged Dictionary
This is the programatic component that has the gathered intelligence and can be used to apply cleaning rules based on abstract rules we specify in the configuration stage.

In [2]:
# Load the metadata/tagged entities
df_tags = get_tagged_entities(coord)

In [3]:
df_tags.head()

,Field Name,Definition,attribute_name,provisional_entity_assignment,static_dynamic,physical_type,logical_type,is_geographical
0,AsOfDate,Date when the data was recorded,AsOfDate,Logical Category,none,unknown,unknown,False
1,Program,Indicator of whether loan was approved under S...,Program,Logical Categories,none,unknown,unknown,False
2,LocationID,SBA's unique lender ID code,LocationID,Conceptual Entity,none,unknown,unknown,False
3,BorrName,Borrower name,BorrName,Person,none,unknown,unknown,False
4,BorrStreet,Borrower street address,BorrStreet,Logical Category,none,unknown,unknown,False


In [4]:
from pathlib import Path
import re

all_columns = df.columns.tolist()
categorical_columns = df_tags[df_tags['logical_type'] == 'categorical']['attribute_name'].tolist()
numerical_columns = df_tags[df_tags['logical_type'] == 'numerical']['attribute_name'].tolist()
datetime_columns = df_tags[df_tags['logical_type'] == 'datetime']['attribute_name'].tolist()

fp_cleaning_recs = "../documents/dd_cleaner/cleaning_recommendations.md"
markdown_text = Path(fp_cleaning_recs).read_text(encoding="utf-8")

def markdown_table_to_df(table_markdown: str) -> pd.DataFrame:
    """Convert a single markdown pipe table block to a DataFrame."""
    lines = [ln.strip() for ln in table_markdown.strip().splitlines() if ln.strip().startswith("|")]
    # Drop the markdown alignment row like |:---|:---|
    lines = [ln for ln in lines if not re.match(r"^\|[:\-\s|]+\|?$", ln)]

    rows = [[cell.strip() for cell in ln.strip("|").split("|")] for ln in lines]
    header, data = rows[0], rows[1:]
    return pd.DataFrame(data, columns=header)

# Extract all contiguous markdown table blocks and combine them
table_blocks = re.findall(r"(?:^\|.*\|\n)+", markdown_text, flags=re.MULTILINE)
df_cleaning_recs = pd.concat(
    [markdown_table_to_df(block) for block in table_blocks],
    ignore_index=True
    )

df_cleaning_recs

,Attribute,Type,Entity,What Needs Fixing,Recommended Fix
0,AsOfDate,unknown,Logical Category,Constant value / Zero variance,drop-attribute
1,Program,unknown,Logical Categories,Constant value / Zero variance,drop-attribute
2,BankNCUANumber,unknown,Logical Category,Extreme sparsity (97.1%): Exceeds null thresho...,drop-attribute
3,ChargeOffDate,unknown,Logical Category,Extreme sparsity (98.2%): Exceeds null thresho...,drop-attribute
4,SoldSecMrktInd,unknown,Logical Category,Constant value / Zero variance,drop-attribute
5,BorrName,unknown,Person,Column contains 0.0% nulls but type is unknown...,user-review
6,BankFDICNumber,unknown,Organization,Column contains 10.6% nulls but type is unknow...,user-review
7,FirstDisbursementDate,unknown,Person,Column contains 18.3% nulls but type is unknow...,user-review
8,InitialInterestRate,unknown,Logical Category,Column contains 0.0% nulls but type is unknown...,user-review
9,FixedorVariableInterestInd,unknown,Logical Category,Column contains 0.0% nulls but type is unknown...,user-review


### Note about drops
The following attribute sets are dropped:
1. The attributes recommended by the tool - but after review
2. The naics description attribute because it is mostly empty
3. The datetime attributes. The rationale for this is that this is a cross-sectional dataset and if I have to use the date information, I need other fields like either the date of last payment or the outstanding balance on the account, perhaps there is some deep business logic to guestimate from the loan-status field, but since this is a demo, I am dropping it rather than including it.
4. Location ID because we are going to use a geo-coding featurization, which is why we have the geographic entity tagging.
5. Paid-in-full and charge-off outcome fields (`paidinfulldate`, `chargeoffdate`, and `grosschargeoffamount`) are explicitly removed as a leakage guard. These attributes reveal the eventual loan outcome and must not be available to the model at training time.


### Loan Condition Definition

Instead of filtering rows out by loan status, derive a new categorical field called `loancondition` so the dataset keeps every record and uses condition as a feature.

| `loancondition` | `loanstatus` examples | Meaning |
| --- | --- | --- |
| `active` | `COMMIT`, `CURRENT`, `IN DEFERMENT` | The loan is live and not yet distressed. |
| `closed` | `PIF`, `PIFLIQ`, `PAID IN FULL`, `PREPAID IN FULL` | The loan has been paid off or otherwise resolved. |
| `distressed` | `CHGOFF`, `1 MONTH PAST DUE`, `IN CATCH-UP`, `PURCH(NOT C/O)` | The loan is delinquent, defaulted, or in loss recovery. |
| `non-origination` | `CANCLD`, `NOTFUND`, `CANCELED`, `NOT FUNDED` | The application never became a live loan. |
| `unknown` | any other status | The status was not recognized by the mapping. |

In [5]:
attributes_rec_for_drop = df_cleaning_recs.loc[
    df_cleaning_recs["Recommended Fix"].str.strip().str.lower() == "drop-attribute",
    ["Attribute", "Recommended Fix"]
]
attributes_rec_for_drop
attributes_to_drop = attributes_rec_for_drop["Attribute"].tolist()
attributes_to_drop += ["naicsdescription", "locationid", "approvalfy", "borrname"]

# Explicit leakage guard: do not retain attributes that record the loan's eventual
# paid-in-full or charge-off outcome. These values are post-outcome signals and
# would leak the target information into the model.
post_outcome_leakage_fields = [
    "paidinfulldate",
    "chargeoffdate",
    "grosschargeoffamount",
]
attributes_to_drop += post_outcome_leakage_fields

datetime_attributes_to_drop = datetime_columns
attributes_to_drop += datetime_attributes_to_drop


In [6]:
attributes_to_drop

['AsOfDate',
 'Program',
 'BankNCUANumber',
 'ChargeOffDate',
 'SoldSecMrktInd',
 'naicsdescription',
 'locationid',
 'approvalfy',
 'borrname',
 'paidinfulldate',
 'chargeoffdate',
 'grosschargeoffamount']

In [7]:
# Normalize the dataframe contract so raw and cleaned files with different casing
# and spacing still work with the notebook's expected lowercase field names.
def canonicalize_dataframe_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy of the dataframe with normalized lowercase column names."""
    def normalize_name(name: object) -> str:
        return re.sub(r"[\s_]+", "", str(name)).lower()
    return df.rename(columns=lambda name: normalize_name(name))


# Loan condition is now derived instead of used to filter rows.
def derive_loancondition(df: pd.DataFrame) -> pd.DataFrame:
    """Add a loancondition column derived from loanstatus."""
    work_df = canonicalize_dataframe_columns(df)
    loanstatus = work_df["loanstatus"].astype("string").str.upper().str.strip()
    loanstatus_key = loanstatus.str.replace(r"[\s\-_/()]+", "", regex=True)

    active_statuses = {"COMMIT", "CURRENT", "INDEFERMENT"}
    closed_statuses = {"PIF", "PIFLIQ", "PAIDINFULL", "PREPAIDINFULL", "PAIDINFULLLIQ"}
    distressed_statuses = {
        "1MONTHPASTDUE",
        "2MONTHSPASTDUE",
        "3MONTHSPASTDUE",
        "4MONTHSPASTDUE",
        "5MONTHSPASTDUE",
        "6MONTHSPASTDUE",
        "7MONTHSPASTDUE",
        "8MONTHSPASTDUE",
        "9MONTHSPASTDUE",
        ">9MONTHSPASTDUE",
        "INCATCHUP",
        "PURCHNOTCO",
        "CHARGEDOFF",
        "ASSETSALE",
        "CHGOFF",
    }
    non_origination_statuses = {"CANCLD", "NOTFUND", "CANCELED", "NOTFUNDED"}

    conditions = [
        loanstatus_key.isin(active_statuses),
        loanstatus_key.isin(closed_statuses),
        loanstatus_key.isin(distressed_statuses),
        loanstatus_key.isin(non_origination_statuses),
    ]
    choices = ["active", "closed", "distressed", "non-origination"]

    work_df["loancondition"] = np.select(conditions, choices, default="unknown")
    return work_df


In [8]:

# Load the metadata/tagged entities
df_tags = get_tagged_entities(coord)

In [9]:
df_tags

,Field Name,Definition,attribute_name,provisional_entity_assignment,static_dynamic,physical_type,logical_type,is_geographical
0,AsOfDate,Date when the data was recorded,AsOfDate,Logical Category,none,unknown,unknown,False
1,Program,Indicator of whether loan was approved under S...,Program,Logical Categories,none,unknown,unknown,False
2,LocationID,SBA's unique lender ID code,LocationID,Conceptual Entity,none,unknown,unknown,False
3,BorrName,Borrower name,BorrName,Person,none,unknown,unknown,False
4,BorrStreet,Borrower street address,BorrStreet,Logical Category,none,unknown,unknown,False
5,BorrCity,Borrower city,BorrCity,Logical Category,none,unknown,unknown,False
6,BorrState,Borrower state,BorrState,Logical Category,none,unknown,unknown,False
7,BorrZip,Borrower zip code,BorrZip,Geographical,none,unknown,unknown,False
8,BankName,Name of the bank that the loan is currently as...,BankName,Logical Category,none,unknown,unknown,False
9,BankFDICNumber,The Federal Depository Insurance Corporation c...,BankFDICNumber,Organization,none,unknown,unknown,False


In [10]:
cat_attrib_filter = df_tags.logical_type == "categorical"
cat_attrib = df_tags[cat_attrib_filter]["Field Name"]

In [11]:
datetime_attrib_filter = df_tags.logical_type == "datetime"
numeric_attrib_filter = df_tags.logical_type == "numeric"
date_time_attrib =  df_tags[datetime_attrib_filter]["Field Name"]
numeric_attrib = df_tags[numeric_attrib_filter]["Field Name"]

In [12]:
def rectify_types(df: pd.DataFrame) -> pd.DataFrame:
    """Coerce selected fields to string-backed categorical in both data and df_tags."""
    global df_tags

    target_columns = {"naicscode", "franchisecode", "congressionaldistrict", "borrzip"}
    normalized_target_columns = {col.replace(" ", "").replace("_", "").lower() for col in target_columns}

    work_df = df.copy()
    normalized_df_columns = {
        column: column.replace(" ", "").replace("_", "").lower()
        for column in work_df.columns
    }

    for column, normalized_column in normalized_df_columns.items():
        if normalized_column in normalized_target_columns:
            work_df[column] = work_df[column].astype("string").astype("category")

    if "attribute_name" in df_tags.columns:
        normalized_attribute_names = (
            df_tags["attribute_name"]
            .astype("string")
            .str.replace(" ", "", regex=False)
            .str.replace("_", "", regex=False)
            .str.lower()
        )
        target_mask = normalized_attribute_names.isin(normalized_target_columns)

        if "logical_type" in df_tags.columns:
            df_tags.loc[target_mask, "logical_type"] = "categorical"

        if "physical_type" in df_tags.columns:
            df_tags.loc[target_mask, "physical_type"] = "str"

    return work_df

In [13]:
def handle_missing_categorical(df: pd.DataFrame) -> pd.DataFrame:
    """Fill missing values in categorical fields with the literal category 'MISSING'."""
    global df_tags

    work_df = df.copy()
    normalized_df_columns = {
        column: column.replace(" ", "").replace("_", "").lower()
        for column in work_df.columns
    }

    categorical_targets = set()
    if "attribute_name" in df_tags.columns and "logical_type" in df_tags.columns:
        categorical_targets = set(
            df_tags.loc[
                df_tags["logical_type"].astype("string").str.lower() == "categorical",
                "attribute_name",
            ]
            .astype("string")
            .str.replace(" ", "", regex=False)
            .str.replace("_", "", regex=False)
            .str.lower()
        )

    for column, normalized_column in normalized_df_columns.items():
        if normalized_column in categorical_targets:
            if isinstance(work_df[column].dtype, pd.CategoricalDtype):
                if "MISSING" not in work_df[column].cat.categories:
                    work_df[column] = work_df[column].cat.add_categories(["MISSING"])
                work_df[column] = work_df[column].fillna("MISSING")
            else:
                work_df[column] = work_df[column].astype("string").fillna("MISSING").astype("category")

    return work_df

In [14]:
def filter_valid_loancondition(df: pd.DataFrame) -> pd.DataFrame:
    """Keep only active, closed, and distressed loancondition values."""
    valid_conditions = {"active", "closed", "distressed"}
    return df.loc[df["loancondition"].isin(valid_conditions)].copy()

In [15]:
def apply_migration(raw_df: pd.DataFrame) -> pd.DataFrame:
    """
    Orchestrate the notebook migration steps in order:
    1. Derive the new loan attribute
    2. Filter loancondition to active/closed/distressed
    3. Rectify the types
    4. Drop the attributes we do not need
    5. Handle missing values for categorical attributes
    """

    work_df = canonicalize_dataframe_columns(raw_df)

    # 1. Derive the new loan attribute
    work_df = derive_loancondition(work_df)

    # 2. Keep only the allowed loan condition classes
    work_df = filter_valid_loancondition(work_df)

    # 3. Rectify the types
    work_df = rectify_types(work_df)

    # 4. Drop the attributes we do not need
    # Explicit leakage guard: remove paid-in-full and charge-off outcome fields before
    # fitting any model. They reveal the eventual loan outcome and can leak the target
    # information into the training data.
    # If first disbursement is unavailable, use approval date as the fallback proxy.
    # This preserves valid loans without breaking the seasoning / loan-age logic.
    work_df['firstdisbursementdate'] = pd.to_datetime(
        work_df['firstdisbursementdate'], errors='coerce'
    ).fillna(pd.to_datetime(work_df['approvaldate'], errors='coerce'))

    work_df = work_df.loc[work_df['firstdisbursementdate'].notnull()]

    fdd = pd.to_datetime(work_df['firstdisbursementdate'], errors='coerce')
    pifd = pd.to_datetime(work_df['paidinfulldate'], errors='coerce')
    work_df = work_df.loc[~(fdd > pifd)]

    work_df = work_df.drop(columns=attributes_to_drop + datetime_attributes_to_drop, errors='ignore')

    # 5. Handle missing values for categorical attributes
    work_df = handle_missing_categorical(work_df)

    return work_df

# Execute the chain
df_raw_baseline = get_cleaned_data(coord)
df_final = apply_migration(df_raw_baseline)

print(f"Final Dataset Shape: {df_final.shape}")


/home/rajiv/programming/kmds_migration/sba_migration/.venv/lib/python3.13/site-packages/dd_cleaner/notebook_utils.py:124: DtypeWarning: Columns (0: ChargeOffDate) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path)


Final Dataset Shape: (75070, 36)


In [16]:
missing_values_report = (
    df_final.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
    .assign(
        missing_pct=lambda d: (d["missing_count"] / len(df_final) * 100).round(2)
    )
    .query("missing_count > 0")
    .sort_values("missing_count", ascending=False)
    .reset_index(names="column")
)

print(f"Columns with unhandled missing values: {missing_values_report.shape[0]}")
missing_values_report

Columns with unhandled missing values: 6


,column,missing_count,missing_pct
0,bankncuanumber,72524,96.61
1,franchisename,65689,87.50
2,soldsecmrktind,52143,69.46
3,bankfdicnumber,6722,8.95
4,businessage,113,0.15
5,businesstype,2,0.00


In [17]:
df_final["loancondition"].value_counts(dropna=False)

loancondition
closed        68177
distressed     6893
Name: count, dtype: int64

In [18]:
counts = df_final["loancondition"].value_counts()
distressed_count = counts.get("distressed", 0)
closed_count = counts.get("closed", 0)
labeled_count = distressed_count + closed_count

imbalance_ratio = distressed_count / closed_count if closed_count else np.nan
bad_share_of_labeled = distressed_count / labeled_count if labeled_count else np.nan

print(f"Imbalance ratio (distressed/closed): {imbalance_ratio:.6f}")
print(f"Bad share among labeled rows (distressed / (distressed + closed)): {bad_share_of_labeled:.6f}")
print(f"Bad share among labeled rows (percent): {bad_share_of_labeled * 100:.2f}%")

imbalance_ratio, bad_share_of_labeled


Imbalance ratio (distressed/closed): 0.101104
Bad share among labeled rows (distressed / (distressed + closed)): 0.025181
Bad share among labeled rows (percent): 2.52%


In [19]:
df_final.columns

Index(['asofdate', 'program', 'borrstreet', 'borrcity', 'borrstate', 'borrzip',
       'bankname', 'bankfdicnumber', 'bankncuanumber', 'bankstreet',
       'bankcity', 'bankstate', 'bankzip', 'grossapproval',
       'sbaguaranteedapproval', 'approvaldate', 'firstdisbursementdate',
       'processingmethod', 'initialinterestrate', 'fixedorvariableinterestind',
       'terminmonths', 'naicscode', 'franchisecode', 'franchisename',
       'projectcounty', 'projectstate', 'sbadistrictoffice',
       'congressionaldistrict', 'businesstype', 'businessage', 'loanstatus',
       'revolverstatus', 'jobssupported', 'collateralind', 'soldsecmrktind',
       'loancondition'],
      dtype='str')

In [20]:
from dd_cleaner.notebook_utils import get_metadata_table, save_user_cleaned_data
save_user_cleaned_data(coord, df_final)

✅ User-cleaned dataset saved to: /home/rajiv/programming/kmds_migration/sba_migration/data/dd_cleaner/sba_loans_raw_user_cleaned.csv


### Save the metadata to the correct location

In [21]:
from dd_cleaner.notebook_utils import get_metadata_table, save_metadata_table
save_metadata_table(coord, df_tags)

✅ Metadata Authority (Expert Overrides) saved to: /home/rajiv/programming/kmds_migration/sba_migration/data/dd_cleaner/sba_loans_raw_metadata_table.csv


## Final Summary: Migration Activities and Next Action

This notebook now has a complete orchestration flow in `apply_migration()` that prepares the dataset for featurization with minimal manual coding.

### What the migration now does end-to-end

1. Derives a new business-ready target attribute (`loancondition`) from `loanstatus`.
2. Filters the dataset to the allowed outcome classes (`active`, `closed`, `distressed`).
3. Rectifies selected fields to string-backed categoricals and synchronizes `df_tags` so metadata remains authoritative.
4. Applies business-rule filters for invalid records (for example, date consistency checks).
5. Drops attributes identified for removal from recommendations and domain review.
6. Handles missing values in categorical attributes using the explicit category `MISSING`.
7. Produces validation outputs including remaining-missing report and class imbalance ratio.

### What has already been automated with coding assistant support

- Boilerplate cleaning logic and reusable helper methods.
- Metadata-aligned type rectification logic (`df_tags` + dataframe consistency).
- Missing value handling and diagnostics/reporting cells.
- Orchestration and reproducible sequencing in a single migration function.

### What remains for the analyst before featurization

1. Review final outputs (`df_final`, missing report, class counts, imbalance ratio).
2. Spot-check edge cases and domain exceptions.
3. Apply corrective overrides only where business context requires it.
4. Freeze this cleaned output as the input contract for the featurization phase.

The key outcome is that the repetitive implementation work is now mostly automated, and the remaining task is high-value review and targeted corrective action before feature engineering.